# Train a tiny GPT on Colab: a healthy band, then three planted bugs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/train-a-tiny-gpt-on-colab/train_tiny_gpt_v1.ipynb)

Companion to https://profrod.ai/articles/train-a-tiny-gpt-on-colab. A 12.3M-parameter GPT in plain PyTorch (no trainer library), trained in float32 on a pinned slice of TinyStories with a 4,096-token BPE tokenizer.

Choose **Runtime > Change runtime type > T4 GPU** (or L4), then run the sections in order. Each one saves its results to `/content/tinygpt/colab-results/` after every run and prints a running total of minutes. If the session disconnects, run the setup cell again and continue: finished runs are skipped, and an unfinished run resumes from its last checkpoint if the machine was not recycled. Set `USE_DRIVE = True` to also copy the results to Google Drive after every run.

| Section | What it runs | Runs | Estimated minutes on a T4 |
|---|---|---|---|
| Recorded | Recomputes the numbers recorded on an Apple M4 Pro; no GPU needed | 0 | under 1 |
| A | Environment, data download and checks, healthy seed 0 | 1 | about 4 to 8 |
| B | Healthy seeds 1 to 4, then this GPU's own 5-seed band | 4 | about 12 to 25 |
| C | Three planted bugs, 3 seeds each, judged against that band | 9 | about 28 to 56 |
| D (optional) | Three more healthy seeds, to count false alarms | 3 | about 9 to 19 |
| Receipt | Assembles the receipt, shows it and offers it as a download | 0 | under 1 |

These are estimates, not measurements. One run is 7.5 × 10¹⁴ floating-point operations (8.2M training tokens at 80.2M FLOPs each, plus the evaluations). A T4's float32 peak is 8.1 TFLOP/s, and the estimate assumes it sustains 25% to 50% of that: 2 to 4 TFLOP/s, which brackets the 2.1 to 2.8 TFLOP/s the Apple M4 Pro reached on its fastest runs here. That gives 3.1 to 6.2 minutes per run, plus about 1 to 2 minutes of setup in section A. If section A's run takes longer than 6 minutes, skip section D. This notebook's receipt replaces these estimates with the T4's own numbers.

In [ ]:
# Setup: pinned dependencies, this folder's code from GitHub, and where results are kept.
!pip -q install "tokenizers==0.23.2"

import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

PRESET = "default"  # "long" trains 3x the steps and takes about 3x the minutes (README: the knob for a larger run)
USE_DRIVE = False  # True: mount Google Drive and copy results there after every run

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/train-a-tiny-gpt-on-colab/"
FILES = ["tinygpt_stats_v1.py", "tinygpt_model_v1.py", "tinygpt_run_v1.py", "data/dataset-v1.json", "data/tokenizer-v1.json",
         "results/runs-v1.jsonl", "results/summary-v1.json", "receipts/tinygpt-v1.json"]
WORK = Path("/content/tinygpt") if Path("/content").exists() else Path("tinygpt").resolve()
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
for name in FILES:
    if not Path(name).exists():
        Path(name).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)

import tinygpt_run_v1 as R  # noqa: E402
import tinygpt_stats_v1 as S  # noqa: E402

OUT = WORK / "colab-results"
OUT.mkdir(exist_ok=True)
RUNS_FILE = OUT / f"runs-colab-{PRESET}-v1.jsonl"
DRIVE = Path("/content/drive/MyDrive/tinygpt-colab-results")
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE.mkdir(parents=True, exist_ok=True)
    for f in DRIVE.glob("*.json*"):  # after a recycled machine, pick up where Drive left off
        if not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)


def rows() -> list[dict]:
    return R.load_runs(RUNS_FILE)


def backup() -> None:
    if USE_DRIVE:
        for f in OUT.glob("*.json*"):
            shutil.copy(f, DRIVE / f.name)


def running_total() -> str:
    mins = sum(r["wallSeconds"] for r in rows()) / 60
    return f"running total: {len(rows())} runs, {mins:.1f} min of training runs"


def run(seed: int, bug: str, purpose: str) -> bool:
    # Train one run unless it already finished; True if it trained now.
    run_id = f"{purpose}-{bug}-s{seed}"
    if run_id in {r["runId"] for r in rows()}:
        print("already finished:", run_id)
        return False
    R.train(seed, bug, purpose, out=RUNS_FILE, ckpt_every=250, resume=True, preset=PRESET)
    backup()
    print(running_total(), flush=True)
    return True


def brief(r: dict) -> dict:
    keys = ("runId", "seed", "bug", "preset", "stepsDone", "divergedAtStep", "initValLoss", "finalValLoss", "tokensPerSecond",
            "medianSecondsPerStep", "memoryBytes", "memoryKind", "started")
    return {**{k: r.get(k) for k in keys}, "minutes": round(r["wallSeconds"] / 60, 2), "trainMinutes": round(r["trainSeconds"] / 60, 2),
            "evalSteps": r["evalSteps"], "valLoss": r["valLoss"]}


def save_part(name: str, data: dict, fresh: bool) -> None:
    # Record a section's receipt part, unless it is already recorded and nothing new ran.
    path = OUT / f"receipt-part-{name}.json"
    if path.exists() and not fresh:
        print(f"section {name} was already recorded; nothing new ran")
        return
    path.write_text(json.dumps(data, indent=2) + "\n")
    backup()


print("working in", WORK, "| preset", PRESET, "|", running_total())

## Recorded: recompute the numbers measured on an Apple M4 Pro

`summarize` rebuilds the summary from `results/runs-v1.jsonl` (every evaluation of every recorded run) with the stats module, and the check compares it with the recorded `results/summary-v1.json`. No GPU is needed.

- **Initial loss.** A fresh model's logits are small and nearly equal, so it predicts about 1/V for every token and the loss is about -ln(1/V) = ln V. With logits that are independent N(0, s²), E[log Σ exp z] is about ln V + s²/2, so the loss starts s²/2 above ln V.
- **The band.** Per evaluation step across the 5 healthy seeds: the mean, a 95% t interval for the mean, and a 95% prediction interval for one new run (mean ± t₀.₉₇₅,₄ · s · √(1 + 1/5)).
- **The rule.** Flag a run once its validation loss is outside the healthy prediction band, on the same side, at 2 evaluations in a row (one evaluation every 50 steps).

In [ ]:
recorded = json.load(open("results/summary-v1.json"))
assert json.loads(json.dumps(R.summarize(write=False))) == recorded, "the recomputed summary differs from the recorded one"


def show(summary: dict, label: str) -> None:
    m, h = summary["model"], summary["healthy"]
    print(f"== {label}: {h['deviceName']}")
    print(f"model: {m['params']:,} parameters; {h['tokensPerSecond']['mean']:,.0f} tokens/s; "
          f"{h['medianSecondsPerStep']['mean'] * 1000:.0f} ms per step; {h['minutes']['mean']:.1f} min per run (mean of 5)")
    il = summary["initialLoss"]
    for seed, measured in il["measured"].items():
        print(f"  initial loss, seed {seed}: measured {measured:.4f}; ln V = {il['lnV']:.4f}; ln V + s^2/2 = {il['predicted'][seed]:.4f}")
    b = h["band"]
    for i, step in enumerate(h["evalSteps"]):
        if step in (0, 50, 100, 200, 500, 1000, 2000, 3000) or step == h["evalSteps"][-1]:
            print(f"  step {step:5d}: mean {b['mean'][i]:.4f}  CI [{b['ciLow'][i]:.4f}, {b['ciHigh'][i]:.4f}]  PI [{b['piLow'][i]:.4f}, {b['piHigh'][i]:.4f}]")
    f = h["finalValLoss"]
    print(f"  final validation loss: mean {f['mean']:.4f}, sd {f['sd']:.4f}, 95% CI {f['ci95'][0]:.4f} to {f['ci95'][1]:.4f}, "
          f"95% PI {f['pi95'][0]:.4f} to {f['pi95'][1]:.4f}")
    fa = summary["falseAlarms"]
    print("  false alarms, held-out healthy controls:", [(c["seed"], c["flag"]) for c in fa["heldOutControls"]])
    print("  false alarms, leave one out:", [(c["seed"], c["flag"]) for c in fa["leaveOneOut"]])
    for name, bug in summary["bugs"].items():
        print(f"  {name}: caught {bug['caught']} of {bug['of']}, flag steps {bug['flagSteps']}, side {bug['sides']}, "
              f"final {[r['finalValLoss'] for r in bug['runs']]}")


show(recorded, "recorded")

In [ ]:
import matplotlib.pyplot as plt


def plot(summary: dict, title: str) -> None:
    h = summary["healthy"]
    steps, b = h["evalSteps"], h["band"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.fill_between(steps, b["piLow"], b["piHigh"], color="0.8", label="healthy 95% prediction band")
    ax.plot(steps, b["mean"], color="0.2", lw=1, label="healthy mean")
    for name, color in zip(summary["bugs"], ("tab:red", "tab:blue", "tab:orange")):
        for k, r in enumerate(summary["bugs"][name]["runs"]):
            ys = [float("nan") if v is None else v for v in r["valLoss"]]
            ax.plot(steps[: len(ys)], ys, color=color, lw=0.8, label=name if k == 0 else None)
    ax.set(xlabel="step", ylabel="validation loss (nats per token)", title=title, ylim=(0, 9))
    ax.legend(frameon=False)
    plt.show()


plot(recorded, "recorded on " + recorded["healthy"]["deviceName"])

## A: environment, data, and healthy seed 0

Downloads the pinned data (about 120 MB), checks its hashes and the tokenizer's, tokenizes it with the committed tokenizer, then trains healthy seed 0 for the preset's steps with a checkpoint every 250 steps.

In [ ]:
import importlib.metadata as md
import platform

import torch

t0 = time.time()
if not torch.cuda.is_available():
    print("No GPU on this runtime: choose Runtime > Change runtime type > T4 GPU. Continuing would time the CPU instead.")
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else None
environment = {
    "device": R.device_name(R.pick_device(None)),
    "nvidiaSmi": smi,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "torch": torch.__version__,
    "tokenizers": md.version("tokenizers"),
    "python": platform.python_version(),
    "cpuCount": os.cpu_count(),
    "preset": PRESET,
    "startedUtc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
print(json.dumps(environment, indent=1))
manifest = R.prepare()  # stops with an error if any hash or token count differs from the pinned one
prepare_minutes = (time.time() - t0) / 60
print(f"data ready and verified in {prepare_minutes:.1f} min")
fresh = run(0, "none", "healthy")
save_part("A", {"environment": environment, "prepareMinutes": round(prepare_minutes, 2),
                "datasetSha256": {p: manifest[p]["sha256"] for p in ("train", "valid")}, "tokenizerSha256": manifest["tokenizer"]["sha256"],
                "runs": [brief(r) for r in rows() if r["runId"] == "healthy-none-s0"], "sectionMinutes": round((time.time() - t0) / 60, 2)}, fresh)
print(f"section A: {(time.time() - t0) / 60:.1f} min")

## B: healthy seeds 1 to 4, and this GPU's band

The same model and data with four more seeds. A seed sets both the initialization and the order of training batches; the validation windows are the same for every run. The band is then built from this GPU's 5 seeds, and each seed is also checked against the band recorded on the M4 Pro: does changing the hardware look like an effect, noise or a bug?

In [ ]:
t0 = time.time()
fresh = [run(seed, "none", "healthy") for seed in (1, 2, 3, 4)]
colab = R.summarize(RUNS_FILE, write=False)
h, rb = colab["healthy"], recorded["healthy"]
cross = []
for r in rows():
    if r["purpose"] == "healthy" and r["evalSteps"] == rb["evalSteps"]:
        flag = S.first_flag(r["valLoss"], rb["band"]["piLow"], rb["band"]["piHigh"], r["evalSteps"])
        inside = rb["finalValLoss"]["pi95"][0] <= r["finalValLoss"] <= rb["finalValLoss"]["pi95"][1]
        cross.append({"seed": r["seed"], "flagAgainstRecordedBand": flag, "finalInsideRecordedPi": inside})
show({**colab, "bugs": {}}, "this runtime")
print("against the recorded M4 Pro band:", cross)
save_part("B", {"runs": [brief(r) for r in rows() if r["purpose"] == "healthy"], "initialLoss": colab["initialLoss"],
                "band": h["band"], "evalSteps": h["evalSteps"], "finalValLoss": h["finalValLoss"], "leaveOneOut": colab["falseAlarms"]["leaveOneOut"],
                "againstRecordedBand": cross, "sectionMinutes": round((time.time() - t0) / 60, 2)}, any(fresh))
print(f"section B: {(time.time() - t0) / 60:.1f} min;", running_total())

## C: three planted bugs, 3 seeds each

Each bug is a one-line change selected by name. `no_causal_mask` skips the line that hides future tokens from attention. `softmax_wrong_dim` normalizes attention over the queries instead of the keys. `no_zero_grad` never clears the gradients, so every step adds to the last. The seeds (10, 11, 12) are ones the band never saw.

In [ ]:
t0 = time.time()
fresh = [run(seed, bug, "bug") for bug in ("no_causal_mask", "softmax_wrong_dim", "no_zero_grad") for seed in (10, 11, 12)]
colab = R.summarize(RUNS_FILE, write=False)
show(colab, "this runtime")
plot(colab, "this runtime: " + colab["healthy"]["deviceName"])
save_part("C", {"runs": [brief(r) for r in rows() if r["purpose"] == "bug"], "rule": colab["falseAlarms"]["rule"],
                "bugs": {k: {kk: vv for kk, vv in v.items() if kk != "runs"} | {"perSeed": [{kk: x[kk] for kk in ("seed", "flag", "finalValLoss", "finalOutsidePi", "divergedAtStep")} for x in v["runs"]]}
                         for k, v in colab["bugs"].items()},
                "sectionMinutes": round((time.time() - t0) / 60, 2)}, any(fresh))
print(f"section C: {(time.time() - t0) / 60:.1f} min;", running_total())

## D (optional): three more healthy seeds, to count false alarms

Healthy runs on seeds 10 to 12, the bug runs' seeds, judged by the same rule. Any flag here is a false alarm. Skip this section if the session is short on time; the leave-one-out check in section B covers the same question with less data.

In [ ]:
t0 = time.time()
fresh = [run(seed, "none", "control") for seed in (10, 11, 12)]
colab = R.summarize(RUNS_FILE, write=False)
print("held-out healthy controls:", [(c["seed"], c["flag"], round(c["finalValLoss"], 4)) for c in colab["falseAlarms"]["heldOutControls"]])
save_part("D", {"runs": [brief(r) for r in rows() if r["purpose"] == "control"], "heldOutControls": colab["falseAlarms"]["heldOutControls"],
                "sectionMinutes": round((time.time() - t0) / 60, 2)}, any(fresh))
print(f"section D: {(time.time() - t0) / 60:.1f} min;", running_total())

## Receipt

Everything above in one JSON file: the environment, every run's minutes and results, this GPU's band, which bugs it caught and when, and the comparison with the recorded M4 Pro band. Send it back as evidence by attaching it to an issue at https://github.com/profrodai/profrodai-resources/issues.

In [ ]:
parts = {p: json.loads((OUT / f"receipt-part-{p}.json").read_text()) for p in "ABCD" if (OUT / f"receipt-part-{p}.json").exists()}
colab = R.summarize(RUNS_FILE, write=False) if rows() else {}
all_rows = rows()
receipt = {
    "experiment": "train-a-tiny-gpt-on-colab v1 (feasibility pilot P1), Colab run",
    "ranOn": sorted({r["started"][:10] for r in all_rows}),
    "environment": parts.get("A", {}).get("environment"),
    "prepareMinutes": parts.get("A", {}).get("prepareMinutes"),
    "preset": PRESET,
    "model": colab.get("model"),
    "dataset": {"revision": R.REVISION, "license": R.LICENSE, **parts.get("A", {}).get("datasetSha256", {}),
                "tokenizerSha256": parts.get("A", {}).get("tokenizerSha256")},
    "runs": [brief(r) for r in all_rows],
    "minutesPerRun": {r["runId"]: round(r["wallSeconds"] / 60, 2) for r in all_rows},
    "totalRunMinutes": round(sum(r["wallSeconds"] for r in all_rows) / 60, 1),
    "sectionMinutes": {p: v.get("sectionMinutes") for p, v in parts.items()},
    "healthy": {k: colab["healthy"][k] for k in ("seeds", "evalSteps", "band", "finalValLoss", "tokensPerSecond", "medianSecondsPerStep",
                                                 "minutes", "memoryBytes", "memoryKind")} if colab.get("healthy") else None,
    "initialLoss": colab.get("initialLoss"),
    "falseAlarms": colab.get("falseAlarms"),
    "bugs": parts.get("C", {}).get("bugs"),
    "againstRecordedBand": parts.get("B", {}).get("againstRecordedBand"),
    "costUsd": 0.0,
    "costNote": "0 on the free Colab tier; if this ran on paid compute units, say how many.",
}
name = OUT / f"tinygpt-colab-receipt-{time.strftime('%Y-%m-%d')}.json"
name.write_text(json.dumps(receipt, indent=2) + "\n")
backup()
print(json.dumps({k: v for k, v in receipt.items() if k not in ("runs", "healthy", "falseAlarms")}, indent=1)[:6000])
h = colab.get("healthy") or {}
caught = "; ".join(f"{k} {v['caught']}/{v['of']} at {v['flagSteps']}" for k, v in (colab.get("bugs") or {}).items())
print(f"\nSUMMARY: {(receipt['environment'] or {}).get('device')}, {len(all_rows)} runs, {receipt['totalRunMinutes']} min of runs; "
      f"healthy final val loss {h.get('finalValLoss', {}).get('mean', float('nan')):.4f} "
      f"(95% CI {h.get('finalValLoss', {}).get('ci95', [float('nan')] * 2)[0]:.4f} to {h.get('finalValLoss', {}).get('ci95', [float('nan')] * 2)[1]:.4f}); "
      f"bugs caught: {caught or 'none run'}")
try:
    from google.colab import files

    files.download(str(name))
except ImportError:
    print("not on Colab; the receipt is at", name)